# Segmentasi Semantik Ikan dengan DeepLabv3 + MobileNetV3

Notebook Google Colab untuk segmentasi biner ikan pada gambar akuarium: piksel ikan menjadi foreground dan sisanya latar belakang. Model memakai DeepLabv3 dengan backbone MobileNetV3-Large yang ringan, sehingga cocok untuk perangkat dengan sumber daya terbatas.

**Sebelum mulai:** aktifkan GPU lewat `Runtime > Change runtime type > T4 GPU`, lalu unggah dataset ke Google Drive (struktur ada di Cell 2) dan jalankan semua cell berurutan.

In [ ]:
#CELL 1 IMPORT LIBRARY

import os
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from torchvision.models.segmentation.deeplabv3 import ASPP

warnings.filterwarnings('ignore')

# Hyperparameter
CONFIG = {
    'image_size': 512,            # gambar & mask di-resize ke 512x512
    'batch_size': 4,
    'epochs': 30,
    'lr': 1e-4,                   # learning rate Adam
    'betas': [0.9, 0.999],
    'weight_decay': 1e-6,
    'val_ratio': 0.2,             # 80% training, 20% validasi
    'seed': 42,
    'alpha': 0.5,                 # bobot BCE pada loss gabungan: alpha*BCE + (1-alpha)*Dice
    'threshold': 0.5,             # ambang biner untuk prediksi
    'atrous_rates': [6, 12, 18],  # laju dilasi ASPP
    'num_workers': 2,
}

# Seed supaya hasil bisa direproduksi
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
torch.cuda.manual_seed_all(CONFIG['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Semua Library Berhasil Diimport")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == 'cuda' else ""))
if device.type == 'cpu':
    print("PERINGATAN: GPU tidak aktif. Training di CPU akan sangat lambat.")

In [ ]:
#CELL 2 HUBUNGKAN DATASET

# Mount Google Drive (di luar Colab, dataset dibaca dari folder kerja saat ini)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive'
except ImportError:
    base_path = '.'

# Setup path dataset
dataset_root = os.path.join(base_path, 'dataset_ikan')
images_path = os.path.join(dataset_root, 'images')
masks_path = os.path.join(dataset_root, 'masks')

# Folder keluaran (model, riwayat training) disimpan di Drive agar tidak hilang
output_dir = os.path.join(base_path, 'fish_segmentation_output')
os.makedirs(output_dir, exist_ok=True)

# Verifikasi struktur dataset
print("Checking dataset structure...")
print(f"Dataset root exists: {os.path.exists(dataset_root)}")
print(f"Images folder exists: {os.path.exists(images_path)}")
print(f"Masks folder exists: {os.path.exists(masks_path)}")

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')


def list_images(directory):
    """Daftar file gambar (terurut) di sebuah folder"""
    return sorted(p for p in Path(directory).iterdir()
                  if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def find_pairs(images_dir, masks_dir, mask_suffix=''):
    """Pasangkan gambar dengan mask berdasarkan nama file (tanpa ekstensi)"""
    masks = {}
    for mask_file in list_images(masks_dir):
        stem = mask_file.stem
        if mask_suffix and stem.endswith(mask_suffix):
            stem = stem[:-len(mask_suffix)]
        masks.setdefault(stem, mask_file)

    pairs, unmatched = [], []
    for image_file in list_images(images_dir):
        if image_file.stem in masks:
            pairs.append((image_file, masks[image_file.stem]))
        else:
            unmatched.append(image_file.name)
    return pairs, unmatched


if not (os.path.exists(images_path) and os.path.exists(masks_path)):
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di {dataset_root}. "
        "Unggah folder 'images' dan 'masks' ke Google Drive (lihat README)."
    )

# Jika nama mask punya akhiran (mis. 'ikan01_mask.png'), isi MASK_SUFFIX = '_mask'
MASK_SUFFIX = ''
pairs, unmatched = find_pairs(images_path, masks_path, MASK_SUFFIX)

print(f"\nJumlah gambar : {len(list_images(images_path))}")
print(f"Jumlah mask   : {len(list_images(masks_path))}")
print(f"Pasangan valid: {len(pairs)}")
if unmatched:
    print(f"PERINGATAN: {len(unmatched)} gambar tanpa mask, contoh: {unmatched[:3]}")
if not pairs:
    raise ValueError("Tidak ada pasangan gambar-mask. Cek nama file atau atur MASK_SUFFIX.")

# Cek isi mask: mask berisi ID kelas (0 = latar, 1+ = kelas objek) yang akan digabung menjadi 'ikan'
sample_mask = Image.open(pairs[0][1])
sample_values = np.unique(np.array(sample_mask))
print(f"\nContoh mask: {pairs[0][1].name} | mode {sample_mask.mode} | nilai piksel {sample_values[:10].tolist()}")
if sample_mask.mode in ('RGB', 'RGBA'):
    print("PERINGATAN: mask berwarna terdeteksi. Pakai folder 'masks' (ID kelas), bukan 'color_masks'.")

In [ ]:
#CELL 3 PREPROCESSING & DATA LOADER

# Normalisasi ImageNet
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def preprocess_image(image, image_size=None):
    """PIL RGB -> tensor [3, H, W] ternormalisasi ImageNet (resize bilinear)"""
    if image_size is not None:
        image = image.resize((image_size, image_size), Image.BILINEAR)
    return TF.normalize(TF.to_tensor(image), IMAGENET_MEAN, IMAGENET_STD)


def load_mask(mask_file):
    """Buka mask. Mask berisi ID kelas (mode L/P) dibaca apa adanya, mask berwarna dijadikan grayscale"""
    mask = Image.open(mask_file)
    if mask.mode not in ('L', 'P', 'I', 'I;16'):
        mask = mask.convert('L')
    return mask


def mask_to_tensor(mask):
    """Mask ID kelas -> tensor biner [1, H, W] bernilai 0/1.
    0 = latar belakang, semua ID kelas > 0 digabung menjadi satu kelas 'ikan'"""
    return torch.from_numpy((np.array(mask) > 0).astype(np.float32)).unsqueeze(0)


def denormalize(image):
    """Tensor ternormalisasi [3, H, W] -> array [H, W, 3] rentang 0-1 (untuk visualisasi)"""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (image.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


class FishSegmentationDataset(Dataset):
    """Dataset segmentasi biner: ikan (1) vs latar belakang (0), tanpa augmentasi"""

    def __init__(self, pairs, image_size=512):
        self.pairs = list(pairs)
        self.image_size = image_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_file, mask_file = self.pairs[index]
        image = Image.open(image_file).convert('RGB')
        mask = load_mask(mask_file)

        size = (self.image_size, self.image_size)
        image = image.resize(size, Image.BILINEAR)
        mask = mask.resize(size, Image.NEAREST)   # nearest agar mask tetap biner
        return preprocess_image(image), mask_to_tensor(mask)


def foreground_ratio(mask_file):
    """Proporsi piksel ikan pada sebuah mask"""
    return mask_to_tensor(load_mask(mask_file)).mean().item()


def stratified_split(pairs, val_ratio, seed, n_bins=5):
    """Split train/validasi terstratifikasi berdasarkan proporsi piksel ikan pada mask.

    Gambar dikelompokkan ke n_bins strata (kuantil proporsi ikan), lalu tiap strata diacak
    dan diambil sesuai rasio validasi sehingga ukuran ikan terwakili di kedua split
    """
    ratios = np.array([foreground_ratio(mask_file) for _, mask_file in pairs])
    edges = np.quantile(ratios, np.linspace(0, 1, n_bins + 1)[1:-1])
    strata = np.digitize(ratios, edges)
    groups = [np.where(strata == s)[0].tolist() for s in range(n_bins)]
    groups = [g for g in groups if g]

    # Jatah validasi per strata (pembagian sisa ke strata dengan pecahan terbesar)
    n_val = min(max(1, round(len(pairs) * val_ratio)), len(pairs) - 1)
    quota = [len(g) * n_val / len(pairs) for g in groups]
    take = [int(q) for q in quota]
    by_fraction = sorted(range(len(groups)), key=lambda j: quota[j] - take[j], reverse=True)
    for j in by_fraction[:n_val - sum(take)]:
        take[j] += 1

    rng = random.Random(seed)
    val_idx = set()
    for group, k in zip(groups, take):
        group = group[:]
        rng.shuffle(group)
        val_idx.update(group[:k])

    train_pairs = [p for i, p in enumerate(pairs) if i not in val_idx]
    val_pairs = [pairs[i] for i in sorted(val_idx)]
    return train_pairs, val_pairs


train_pairs, val_pairs = stratified_split(pairs, CONFIG['val_ratio'], CONFIG['seed'])

train_dataset = FishSegmentationDataset(train_pairs, CONFIG['image_size'])
val_dataset = FishSegmentationDataset(val_pairs, CONFIG['image_size'])

# Cabang global pooling ASPP memakai BatchNorm: batch berukuran 1 saat training akan error,
# jadi batch sisa seperti itu dibuang
drop_last = len(train_dataset) % CONFIG['batch_size'] == 1
loader_kwargs = dict(batch_size=CONFIG['batch_size'], num_workers=CONFIG['num_workers'],
                     pin_memory=(device.type == 'cuda'))
train_loader = DataLoader(train_dataset, shuffle=True, drop_last=drop_last, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

print(f"Data training  : {len(train_dataset)} gambar")
print(f"Data validasi  : {len(val_dataset)} gambar")

# Tampilkan contoh hasil preprocessing
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row, idx in enumerate(random.Random(0).sample(range(len(train_dataset)), 3)):
    image, mask = train_dataset[idx]
    fish = mask[0].numpy() > 0.5
    overlay = denormalize(image).copy()
    overlay[fish] = 0.5 * overlay[fish] + 0.5 * np.array([0, 1, 0])
    for col, (data, title) in enumerate([(denormalize(image), 'Gambar (512x512)'),
                                         (mask[0].numpy(), 'Mask (0/1)'),
                                         (overlay, 'Overlay')]):
        axes[row, col].imshow(data, cmap='gray' if data.ndim == 2 else None)
        axes[row, col].set_title(title)
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#CELL 4 MODEL: DEEPLABV3 + MOBILENETV3-LARGE

class FishSegNet(nn.Module):
    """DeepLabv3 dengan backbone MobileNetV3-Large dan decoder skip connection.

    Alur: backbone (pretrained ImageNet, dilated) -> ASPP -> upsampling bilinear ->
    gabung dengan fitur level rendah backbone (skip connection) -> konvolusi refinement ->
    konvolusi 1x1 dengan 1 kanal keluaran (logit; sigmoid diterapkan di luar model)
    """

    def __init__(self, pretrained_backbone=True, atrous_rates=(6, 12, 18),
                 decoder_channels=256, low_level_channels=48):
        super().__init__()
        weights = MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained_backbone else None
        features = mobilenet_v3_large(weights=weights, dilated=True).features

        # Backbone dibelah dua: fitur level rendah (stride 4) dan fitur level tinggi (stride 16)
        self.low_level = features[:4]
        self.high_level = features[4:]

        # ASPP: konvolusi atrous paralel + global average pooling + konvolusi 1x1,
        # hasilnya digabung (concatenate) lalu diproyeksikan ke decoder_channels
        self.aspp = ASPP(features[-1].out_channels, list(atrous_rates), decoder_channels)

        # Skip connection dari fitur level rendah
        self.low_project = nn.Sequential(
            nn.Conv2d(features[3].out_channels, low_level_channels, 1, bias=False),
            nn.BatchNorm2d(low_level_channels),
            nn.ReLU(inplace=True),
        )

        # Konvolusi refinement untuk memperjelas batas objek
        self.refine = nn.Sequential(
            nn.Conv2d(decoder_channels + low_level_channels, decoder_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(decoder_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_channels, decoder_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(decoder_channels),
            nn.ReLU(inplace=True),
        )

        # Output layer: konvolusi 1x1, 1 kanal (segmentasi biner)
        self.head = nn.Conv2d(decoder_channels, 1, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[-2:]
        low = self.low_level(x)
        high = self.high_level(low)

        out = self.aspp(high)
        out = F.interpolate(out, size=low.shape[-2:], mode='bilinear', align_corners=False)
        out = self.refine(torch.cat([out, self.low_project(low)], dim=1))
        out = self.head(out)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=False)


print("Creating DeepLabv3 + MobileNetV3-Large model...")
model = FishSegNet(pretrained_backbone=True, atrous_rates=CONFIG['atrous_rates']).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Jumlah parameter: {n_params:,}")

# Cek bentuk keluaran
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CONFIG['image_size'], CONFIG['image_size']).to(device)
    print(f"Input  : {tuple(dummy.shape)}")
    print(f"Output : {tuple(model(dummy).shape)}")

In [ ]:
#CELL 5 LOSS & METRIK EVALUASI

class BCEDiceLoss(nn.Module):
    """L = alpha * BCE + (1 - alpha) * Dice loss, dengan Dice loss = 1 - Dice"""

    def __init__(self, alpha=0.5, eps=1e-7):
        super().__init__()
        self.alpha = alpha
        self.eps = eps

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        probs = torch.sigmoid(logits).flatten(1)
        t = targets.flatten(1)
        inter = (probs * t).sum(dim=1)
        dice = (2 * inter + self.eps) / (probs.sum(dim=1) + t.sum(dim=1) + self.eps)
        return self.alpha * bce + (1 - self.alpha) * (1 - dice).mean()


METRIC_NAMES = ('dice', 'iou', 'pixel_accuracy', 'precision', 'recall', 'f1')


def confusion_counts(preds, targets):
    """TP/FP/FN/TN per gambar dari mask biner [N, 1, H, W]"""
    p = preds.flatten(1).float()
    t = targets.flatten(1).float()
    return {'tp': (p * t).sum(1), 'fp': (p * (1 - t)).sum(1),
            'fn': ((1 - p) * t).sum(1), 'tn': ((1 - p) * (1 - t)).sum(1)}


def metrics_from_counts(c, eps=1e-7):
    """Dice = 2|X∩Y| / (|X|+|Y|), IoU = |X∩Y| / |X∪Y|, Pixel Accuracy, Precision, Recall, F1 per gambar"""
    tp, fp, fn, tn = c['tp'], c['fp'], c['fn'], c['tn']
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    return {
        'dice': (2 * tp + eps) / (2 * tp + fp + fn + eps),
        'iou': (tp + eps) / (tp + fp + fn + eps),
        'pixel_accuracy': (tp + tn) / (tp + tn + fp + fn),
        'precision': precision,
        'recall': recall,
        'f1': 2 * precision * recall / (precision + recall + eps),
    }


class MetricTracker:
    """Mengumpulkan metrik per gambar + total confusion selama evaluasi"""

    def __init__(self):
        self.values = {name: [] for name in METRIC_NAMES}
        self.totals = {'tp': 0.0, 'fp': 0.0, 'fn': 0.0, 'tn': 0.0}

    def update(self, preds, targets):
        counts = confusion_counts(preds, targets)
        for name, value in metrics_from_counts(counts).items():
            self.values[name].append(value.double().cpu().numpy())
        for name, value in counts.items():
            self.totals[name] += float(value.double().sum())

    def per_image(self):
        return {name: np.concatenate(parts) for name, parts in self.values.items()}

    def summary(self):
        return {name: float(v.mean()) for name, v in self.per_image().items()}

    def confusion_percent(self):
        total = sum(self.totals.values())
        return {name: 100.0 * v / total for name, v in self.totals.items()}


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    """Evaluasi model pada loader. Mengembalikan (ringkasan metrik, tracker)"""
    model.eval()
    tracker = MetricTracker()
    total_loss, total_samples = 0.0, 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        if criterion is not None:
            total_loss += criterion(logits, masks).item() * images.size(0)
        total_samples += images.size(0)
        tracker.update(torch.sigmoid(logits) > CONFIG['threshold'], masks > 0.5)

    summary = tracker.summary()
    if criterion is not None:
        summary['loss'] = total_loss / total_samples
    return summary, tracker


criterion = BCEDiceLoss(alpha=CONFIG['alpha'])
print("Loss dan metrik siap dipakai")

In [ ]:
#CELL 6 TRAINING MODEL

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'],
                             betas=tuple(CONFIG['betas']),
                             weight_decay=CONFIG['weight_decay'])

best_model_path = os.path.join(output_dir, 'best_model.pt')
history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': [], 'val_iou': []}
best_dice = -1.0

print(f"Training {CONFIG['epochs']} epoch pada {device}...\n")
for epoch in range(1, CONFIG['epochs'] + 1):
    start = time.time()

    # Training
    model.train()
    train_tracker = MetricTracker()
    running_loss, seen = 0.0, 0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        seen += images.size(0)
        with torch.no_grad():
            train_tracker.update(torch.sigmoid(logits) > CONFIG['threshold'], masks > 0.5)
    train_loss = running_loss / seen
    train_dice = train_tracker.summary()['dice']

    # Validasi
    val_metrics, _ = evaluate(model, val_loader, criterion)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_metrics['dice'])
    history['val_iou'].append(val_metrics['iou'])

    # Simpan model terbaik (Dice validasi tertinggi) ke Google Drive
    improved = val_metrics['dice'] > best_dice
    if improved:
        best_dice = val_metrics['dice']
        torch.save({'model_state': model.state_dict(), 'config': CONFIG, 'epoch': epoch,
                    'metrics': val_metrics}, best_model_path)

    print(f"Epoch {epoch:>2}/{CONFIG['epochs']} | train loss {train_loss:.4f} | "
          f"val loss {val_metrics['loss']:.4f} | train dice {train_dice:.4f} | "
          f"val dice {val_metrics['dice']:.4f} | {time.time() - start:.1f}s"
          + (" *" if improved else ""))

with open(os.path.join(output_dir, 'history.json'), 'w') as f:
    json.dump(history, f, indent=2)

print(f"\nTraining selesai. Dice validasi terbaik: {best_dice:.4f}")
print(f"Model terbaik disimpan di: {best_model_path}")

In [ ]:
#CELL 7 KURVA TRAINING

epochs_range = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(11, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Training Loss')
plt.plot(epochs_range, history['val_loss'], label='Validation Loss')
plt.title('Loss (BCE + Dice)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Dice
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_dice'], label='Training Dice')
plt.plot(epochs_range, history['val_dice'], label='Validation Dice')
plt.title('Dice Score')
plt.xlabel('Epoch')
plt.ylabel('Dice')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
#CELL 8 EVALUASI MODEL TERBAIK

# Muat checkpoint terbaik
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state'])
print(f"Memuat model terbaik dari epoch {checkpoint['epoch']}")

val_summary, tracker = evaluate(model, val_loader, criterion)
per_image = tracker.per_image()
confusion = tracker.confusion_percent()

LABELS = {'dice': 'Dice Score', 'iou': 'IoU', 'pixel_accuracy': 'Pixel Accuracy',
          'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1 Score'}

print(f"\nEvaluasi pada {len(val_dataset)} gambar validasi\n")
print(f"{'Metrik':<16}{'Nilai':>10}")
print("-" * 26)
for name in METRIC_NAMES:
    print(f"{LABELS[name]:<16}{val_summary[name]:>10.4f}")
print(f"\nValidation loss: {val_summary['loss']:.4f}")

In [ ]:
#CELL 9 DISTRIBUSI METRIK & CONFUSION MATRIX

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Histogram metrik per gambar
for ax, (key, title) in zip(axes[:3], [('dice', 'Dice Score'), ('iou', 'IoU'),
                                       ('pixel_accuracy', 'Pixel Accuracy')]):
    values = per_image[key]
    ax.hist(values, bins=20, range=(0, 1), color='tab:blue', edgecolor='white')
    ax.axvline(values.mean(), color='tab:red', linestyle='--', label=f'rata-rata {values.mean():.4f}')
    ax.set_title(f'Distribusi {title}')
    ax.set_xlabel('Nilai')
    ax.set_ylabel('Jumlah gambar')
    ax.legend()

# Confusion matrix piksel dalam persen dari total piksel
matrix = np.array([[confusion['tn'], confusion['fp']],
                   [confusion['fn'], confusion['tp']]])
axes[3].imshow(matrix, cmap='Blues')
axes[3].set_xticks([0, 1])
axes[3].set_xticklabels(['Latar', 'Ikan'])
axes[3].set_yticks([0, 1])
axes[3].set_yticklabels(['Latar', 'Ikan'])
axes[3].set_xlabel('Prediksi')
axes[3].set_ylabel('Sebenarnya')
axes[3].set_title('Confusion Matrix (% piksel)')
for (r, c), value in np.ndenumerate(matrix):
    axes[3].text(c, r, f'{value:.3f}%', ha='center', va='center',
                 color='white' if value > matrix.max() / 2 else 'black')

plt.tight_layout()
plt.show()

print(f"Gambar dengan Dice > 0.7: {(per_image['dice'] > 0.7).mean() * 100:.1f}%")
print(f"False negative: {confusion['fn']:.3f}% | False positive: {confusion['fp']:.3f}%")

In [ ]:
#CELL 10 VISUALISASI KASUS TERBAIK DAN TERBURUK

@torch.no_grad()
def show_cases(indices, titles, suptitle):
    """Tampilkan gambar, mask asli, prediksi, dan overlay.
    Overlay: hijau = benar (TP), merah = salah prediksi ikan (FP), biru = ikan terlewat (FN)"""
    model.eval()
    fig, axes = plt.subplots(len(indices), 4, figsize=(12, 3 * len(indices)), squeeze=False)
    for row, (idx, title) in enumerate(zip(indices, titles)):
        image, mask = val_dataset[idx]
        prob = torch.sigmoid(model(image.unsqueeze(0).to(device)))[0, 0].cpu().numpy()
        pred = prob > CONFIG['threshold']
        gt = mask[0].numpy() > 0.5
        img = denormalize(image)

        overlay = img.copy()
        for region, color in [(gt & pred, [0, 1, 0]), (~gt & pred, [1, 0, 0]), (gt & ~pred, [0, 0, 1])]:
            overlay[region] = 0.5 * overlay[region] + 0.5 * np.array(color)

        for col, (data, label) in enumerate([(img, title), (gt.astype(float), 'Mask asli'),
                                             (pred.astype(float), 'Prediksi'), (overlay, 'Overlay')]):
            axes[row, col].imshow(data, cmap='gray' if data.ndim == 2 else None, vmin=0, vmax=1)
            axes[row, col].set_title(label, fontsize=9)
            axes[row, col].axis('off')
    fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


names = [p[0].name for p in val_dataset.pairs]
rank = np.argsort(per_image['dice'])
k = min(4, len(val_dataset))

best_idx = [int(i) for i in rank[::-1][:k]]
worst_idx = [int(i) for i in rank[:k]]

show_cases(best_idx, [f"{names[i]} (Dice {per_image['dice'][i]:.3f})" for i in best_idx],
           'Kasus terbaik')
show_cases(worst_idx, [f"{names[i]} (Dice {per_image['dice'][i]:.3f})" for i in worst_idx],
           'Kasus terburuk')

In [ ]:
#CELL 11 VISUALISASI AKTIVASI LAYER

@torch.no_grad()
def show_activations(index):
    """Heatmap aktivasi (rata-rata nilai absolut semua kanal) pada layer awal, tengah, dan akhir"""
    model.eval()
    activations = {}

    def save_to(name):
        def hook(module, inputs, output):
            activations[name] = output
        return hook

    hooks = [
        model.low_level.register_forward_hook(save_to('Layer awal (backbone level rendah)')),
        model.high_level.register_forward_hook(save_to('Layer tengah (backbone level tinggi)')),
        model.refine.register_forward_hook(save_to('Layer akhir (decoder)')),
    ]
    image, _ = val_dataset[index]
    model(image.unsqueeze(0).to(device))
    for h in hooks:
        h.remove()

    base = denormalize(image)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(base)
    axes[0].set_title(names[index])
    for ax, (title, act) in zip(axes[1:], activations.items()):
        heat = act[0].abs().mean(dim=0, keepdim=True).unsqueeze(0)
        heat = F.interpolate(heat, size=base.shape[:2], mode='bilinear', align_corners=False)[0, 0]
        heat = heat.cpu().numpy()
        heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
        ax.imshow(base)
        ax.imshow(heat, cmap='jet', alpha=0.5)
        ax.set_title(title, fontsize=9)
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


show_activations(best_idx[0])
show_activations(worst_idx[0])

In [ ]:
#CELL 12 WAKTU INFERENSI

@torch.no_grad()
def measure_inference_time(model, dataset, warmup=3):
    """Waktu inferensi per gambar (batch 1, hanya forward pass model) dalam milidetik"""
    model.eval()
    first = dataset[0][0].unsqueeze(0).to(device)
    for _ in range(warmup):
        model(first)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    times = []
    for i in range(len(dataset)):
        image = dataset[i][0].unsqueeze(0).to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        model(image)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)
    return np.array(times)


times_ms = measure_inference_time(model, val_dataset)
print(f"Device                   : {device}")
print(f"Rata-rata waktu inferensi: {times_ms.mean():.2f} ms/gambar (std {times_ms.std():.2f})")
print("Catatan: hanya forward pass model, tanpa praproses. Hasil bergantung pada hardware.")

In [ ]:
#CELL 13 PREDIKSI GAMBAR BARU

@torch.no_grad()
def predict_image(image_path, show=True):
    """Prediksi mask ikan untuk satu gambar. Mask dikembalikan pada ukuran gambar asli"""
    model.eval()
    image = Image.open(image_path).convert('RGB')
    width, height = image.size

    x = preprocess_image(image, CONFIG['image_size']).unsqueeze(0).to(device)
    probs = torch.sigmoid(model(x))
    probs = F.interpolate(probs, size=(height, width), mode='bilinear', align_corners=False)
    mask = (probs[0, 0] > CONFIG['threshold']).cpu().numpy()

    if show:
        overlay = np.array(image, dtype=np.float32)
        overlay[mask] = 0.5 * overlay[mask] + 0.5 * np.array([0, 255, 0])
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        axes[0].imshow(image)
        axes[0].set_title('Gambar')
        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title('Mask prediksi')
        axes[2].imshow(overlay.astype(np.uint8))
        axes[2].set_title(f'Overlay ({100 * mask.mean():.1f}% piksel = ikan)')
        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
    return mask


# Contoh: 3 gambar acak dari data validasi
for image_file, _ in random.sample(val_dataset.pairs, min(3, len(val_dataset))):
    print(image_file.name)
    predict_image(str(image_file))

# Untuk gambar sendiri (unggah ke Drive atau Colab), ganti path lalu jalankan:
# predict_image('/content/drive/MyDrive/foto_ikan.jpg')

# PENJELASAN SINGKAT

Cell 1: Import Library
Mengimpor PyTorch, torchvision, dan library pendukung. Mengatur hiperparameter (input 512x512, batch 4, 30 epoch, Adam lr 1e-4), seed, dan device (GPU/CPU).

Cell 2: Koneksi Dataset
Menghubungkan Google Drive, mengatur path dataset (`dataset_ikan/images` dan `dataset_ikan/masks`), lalu memasangkan gambar dengan mask berdasarkan nama file.

Cell 3: Preprocessing & Data Loader
Resize ke 512x512 (bilinear untuk gambar, nearest untuk mask), normalisasi ImageNet, mask ID kelas dikonversi ke tensor biner (semua ID > 0 digabung menjadi ikan). Data dibagi 80% training dan 20% validasi secara terstratifikasi berdasarkan proporsi piksel ikan pada mask.

Cell 4: Model
DeepLabv3 dengan backbone MobileNetV3-Large (pretrained ImageNet). ASPP (laju dilasi 6, 12, 18, global average pooling, konvolusi 1x1) diikuti decoder: upsampling bilinear, skip connection dari fitur level rendah, konvolusi refinement, dan konvolusi 1x1 dengan 1 kanal keluaran.

Cell 5: Loss dan Metrik
Loss gabungan alpha * BCE + (1 - alpha) * Dice loss. Metrik Dice, IoU, Pixel Accuracy, Precision, Recall, dan F1 dihitung per gambar lalu dirata-ratakan.

Cell 6: Training
Melatih model selama 30 epoch dan menyimpan checkpoint dengan Dice validasi tertinggi ke Google Drive (`fish_segmentation_output/best_model.pt`).

Cell 7: Kurva Training
Grafik loss serta Dice training dan validasi per epoch.

Cell 8: Evaluasi
Memuat model terbaik dan menghitung semua metrik pada data validasi.

Cell 9: Distribusi Metrik dan Confusion Matrix
Histogram Dice, IoU, Pixel Accuracy antar gambar, dan confusion matrix piksel dalam persen.

Cell 10: Kasus Terbaik dan Terburuk
Visualisasi gambar, mask asli, prediksi, dan overlay untuk 4 gambar dengan Dice tertinggi dan terendah.

Cell 11: Aktivasi Layer
Heatmap aktivasi pada layer awal, tengah, dan akhir model untuk satu kasus terbaik dan satu kasus terburuk.

Cell 12: Waktu Inferensi
Mengukur rata-rata waktu forward pass per gambar (batch 1).

Cell 13: Prediksi Gambar Baru
Fungsi `predict_image` untuk memprediksi mask ikan pada gambar apa pun, lengkap dengan overlay.

**Catatan:** data validasi dipakai juga untuk memilih checkpoint terbaik, sehingga hasilnya cenderung sedikit optimistis. Hasil dapat berbeda antar run karena split data, seed, dan hardware.